In [ ]:
import json
from pathlib import Path


json_file = r"D:\DMSc_Dissertation_Project\datasets\ACDC\gt_detection_trainval\gt_detection\fog\instancesonly_fog_train_gt_detection.json"

output_labels = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\labels\train"



Path(output_labels).mkdir(parents=True, exist_ok=True)

with open(json_file, "r") as f:
    coco = json.load(f)

# COCO IDs -> YOLO IDs
category_map = {
    24:0,   # person
    25:1,   # rider
    26:2,   # car
    27:3,   # truck
    28:4,   # bus
    31:5,   # train
    32:6,   # motorcycle
    33:7    # bicycle
}

images = {}

for img in coco["images"]:
    images[img["id"]] = img

labels = {}

for ann in coco["annotations"]:

    if ann["category_id"] not in category_map:
        continue

    img = images[ann["image_id"]]

    w = img["width"]
    h = img["height"]

    x,y,bw,bh = ann["bbox"]

    xc = (x + bw/2)/w
    yc = (y + bh/2)/h
    bw /= w
    bh /= h

    cls = category_map[ann["category_id"]]

    name = Path(img["file_name"]).stem

    labels.setdefault(name,[])

    labels[name].append(
        f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
    )

for name,lines in labels.items():

    with open(Path(output_labels)/f"{name}.txt","w") as f:
        f.write("\n".join(lines))

print("Finished!")
print("Images:",len(images))
print("Labels:",len(labels))

In [ ]:
import json
from pathlib import Path


json_file = r"D:\DMSc_Dissertation_Project\datasets\ACDC\gt_detection_trainval\gt_detection\fog\instancesonly_fog_val_gt_detection.json"

output_labels = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\labels\val"

Path(output_labels).mkdir(parents=True, exist_ok=True)

with open(json_file, "r") as f:
    coco = json.load(f)

# COCO IDs -> YOLO IDs
category_map = {
    24:0,   # person
    25:1,   # rider
    26:2,   # car
    27:3,   # truck
    28:4,   # bus
    31:5,   # train
    32:6,   # motorcycle
    33:7    # bicycle
}

images = {}

for img in coco["images"]:
    images[img["id"]] = img

labels = {}

for ann in coco["annotations"]:

    if ann["category_id"] not in category_map:
        continue

    img = images[ann["image_id"]]

    w = img["width"]
    h = img["height"]

    x,y,bw,bh = ann["bbox"]

    xc = (x + bw/2)/w
    yc = (y + bh/2)/h
    bw /= w
    bh /= h

    cls = category_map[ann["category_id"]]

    name = Path(img["file_name"]).stem

    labels.setdefault(name,[])

    labels[name].append(
        f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
    )

for name,lines in labels.items():

    with open(Path(output_labels)/f"{name}.txt","w") as f:
        f.write("\n".join(lines))

print("Finished!")
print("Images:",len(images))
print("Labels:",len(labels))

In [ ]:
from pathlib import Path

train_imgs = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\images\train").glob("*.png")}
train_lbls = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\labels\train").glob("*.txt")}

print("Train matched:", len(train_imgs & train_lbls))

val_imgs = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\images\val").glob("*.png")}
val_lbls = {p.stem for p in Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\labels\val").glob("*.txt")}

print("Val matched:", len(val_imgs & val_lbls))

In [ ]:
from pathlib import Path

print("Train images :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\images\train").glob("*.png"))))
print("Train labels :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\labels\train").glob("*.txt"))))

print("Val images :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\images\val").glob("*.png"))))
print("Val labels :", len(list(Path(r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\labels\val").glob("*.txt"))))

In [ ]:
from ultralytics.data.utils import check_det_dataset

check_det_dataset(
    r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\acdc_fog.yaml"
)

In [ ]:
from ultralytics import YOLO

def main():

    yaml_path = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG\acdc_fog.yaml"

    model = YOLO("yolov8l.pt")

    print("----- Training Started -----")

    model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=4,
        device=0,
        workers=4,
        project="ACDC_Fog_Project",
        name="train_yolov8l_fog"
    )

    print("----- Training Completed -----")


if __name__ == "__main__":
    main()

In [ ]:
import shutil
import os
from datetime import datetime

# 1. Define your exact source and new destination paths
source_dir = r"C:\Users\Varis\runs\detect\ACDC_Fog_Project\train_yolov8l_fog-5"
target_parent_dir = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG"

# Creating a timestamped folder name to avoid overwriting previous runs
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
destination_dir = os.path.join(target_parent_dir, f"train_backup_{timestamp}")

try:
    print("Starting the backup process...")
    # 2. Copy the entire folder structure
    shutil.copytree(source_dir, destination_dir)
    print(f"🎉 Backup Successful! Your files are saved at:\n--> {destination_dir}")
except FileNotFoundError:
    print("Error: The source folder was not found. Please verify your C: drive path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os

# Your terminal validation log string
terminal_output = """
Validating C:\\Users\\Varis\\runs\\detect\\ACDC_Fog_Project\\train_yolov8l_fog-5\\weights\\best.pt...
Ultralytics 8.4.81  Python-3.13.3 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 113 layers, 43,612,776 parameters, 0 gradients, 164.8 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all        100        640      0.643      0.482      0.532      0.335
                person         36         62      0.681      0.452      0.491      0.238
                 rider         10         11      0.793      0.273      0.464      0.218
                   car         99        496      0.687      0.768       0.79      0.571
                 truck         33         41      0.562      0.281      0.338      0.228
                   bus          5          5          1       0.78      0.829      0.632
                 train          3          3      0.335      0.667      0.665      0.409
            motorcycle         10         11      0.397      0.273      0.224      0.147
               bicycle         10         11       0.69      0.364      0.457      0.238
Speed: 0.4ms preprocess, 13.3ms inference, 0.0ms loss, 2.6ms postprocess per image
"""

# Destination directory path
target_directory = r"D:\DMSc_Dissertation_Project\datasets\ACDC\YOLO_FOG"

# Create the directory if it doesn't exist yet
if not os.path.exists(target_directory):
    os.makedirs(target_directory)

file_path = os.path.join(target_directory, "validation_terminal_output.txt")

# Write the file down
with open(file_path, "w", encoding="utf-8") as f:
    f.write(terminal_output.strip())

print(f"Terminal output text file successfully saved to:\n--> {file_path}")